In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [3]:
load_dotenv()
model = ChatGoogleGenerativeAI(
    model='gemini-3.1-flash-lite',
    google_api_key=os.getenv('GEMINI_API_KEY')
)

In [4]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [5]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [6]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [7]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()


In [8]:
intial_state = {'title': 'about Quantiphi'}

final_state = workflow.invoke(intial_state)

print(final_state)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'title': 'about Quantiphi', 'outline': [{'type': 'text', 'text': 'To write an effective blog post about **Quantiphi**, you need to balance technical credibility with business value. Quantiphi is a "born-in-the-cloud" AI-first digital engineering company, and your blog should reflect their reputation for solving complex data problems.\n\nHere is a detailed outline for a blog post titled: **"Quantiphi: Driving Business Transformation Through Applied AI and Data Engineering."**\n\n---\n\n### **Title Options:**\n*   *Quantiphi: The AI-First Engine Powering Enterprise Digital Transformation*\n*   *Decoding Quantiphi: How They Are Solving Complex Problems with Applied AI*\n*   *Beyond the Buzz: Inside Quantiphi’s Approach to Data and Analytics*\n\n---\n\n### **1. Introduction**\n*   **The Hook:** Briefly discuss the current AI gold rush—how many companies have the "what" (tools) but lack the "how" (execution).\n*   **Introducing Quantiphi:** Define who they are: An AI-first digital engineer

In [9]:
print(final_state['outline'])

[{'type': 'text', 'text': 'To write an effective blog post about **Quantiphi**, you need to balance technical credibility with business value. Quantiphi is a "born-in-the-cloud" AI-first digital engineering company, and your blog should reflect their reputation for solving complex data problems.\n\nHere is a detailed outline for a blog post titled: **"Quantiphi: Driving Business Transformation Through Applied AI and Data Engineering."**\n\n---\n\n### **Title Options:**\n*   *Quantiphi: The AI-First Engine Powering Enterprise Digital Transformation*\n*   *Decoding Quantiphi: How They Are Solving Complex Problems with Applied AI*\n*   *Beyond the Buzz: Inside Quantiphi’s Approach to Data and Analytics*\n\n---\n\n### **1. Introduction**\n*   **The Hook:** Briefly discuss the current AI gold rush—how many companies have the "what" (tools) but lack the "how" (execution).\n*   **Introducing Quantiphi:** Define who they are: An AI-first digital engineering company that solves complex, data-dr

In [10]:
print(final_state['content'])

[{'type': 'text', 'text': '# Quantiphi: The AI-First Engine Powering Enterprise Digital Transformation\n\nWe are currently living through an "AI Gold Rush." Every enterprise is scrambling to integrate artificial intelligence into their operations, but there is a glaring disconnect in the market: most organizations have access to the *tools* (the "what"), but they are struggling with the *execution* (the "how").\n\nIn this environment, bridging the gap between raw data and tangible business value has become the defining challenge of the decade. Enter **Quantiphi**, an AI-first digital engineering company that has spent over a decade proving that complex, data-driven problems are not just solvable—they are the key to competitive differentiation.\n\n---\n\n### Who is Quantiphi? The Foundation\nFounded in 2013, Quantiphi was "born in the cloud" and built with a singular, unwavering focus: **Applied AI.** While many legacy IT firms spent years retrofitting AI onto outdated architectures, Qu